# Construcción de series temporales semanales de dengue y covariables meteorológicas

El objetivo de este notebook es construir las series temporales que servirán como base para las etapas posteriores de **clustering dinámico** y **pronóstico probabilístico con covariables exógenas**.

A partir de los registros individuales de casos confirmados de dengue se generan dos representaciones semanales para cada entidad federativa:

$$
Y^{(\mathrm{casos})}_{s,t} =
\text{número de casos confirmados en el estado } s
\text{ durante la semana } t,
$$

y

$$
Y^{(\mathrm{inc})}_{s,t} =
\frac{Y^{(\mathrm{casos})}_{s,t}}
{N_{s,a(t)}}\times 100,000,
$$

donde $N_{s,a(t)}$ es la población del estado $s$ en el año correspondiente a la semana $t$.

Además, se descargan covariables meteorológicas diarias de NASA POWER y se transforman a frecuencia semanal. El resultado final es un conjunto de matrices alineadas en tiempo y espacio:

$$
Y_{s,t},
\qquad
X^{(1)}_{s,t},
\ldots,
X^{(p)}_{s,t},
$$

donde las filas representan semanas y las columnas corresponden a las 32 entidades federativas.

El flujo general del notebook es:

$$
\boxed{
\text{Registros individuales de dengue}
}
$$

$$
\downarrow
$$

$$
\boxed{
\text{Casos e incidencia semanal por estado}
}
$$

$$
\downarrow
$$

$$
\boxed{
\text{Descarga diaria de covariables meteorológicas}
}
$$

$$
\downarrow
$$

$$
\boxed{
\text{Agregación semanal y alineación temporal}
}
$$

$$
\downarrow
$$

$$
\boxed{
\text{Matrices para clustering y pronóstico}
}
$$

In [43]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import geopandas as gpd
import unicodedata
import re

In [44]:
BASE = Path("datos_dengue_historicos")

DATA_DIR = BASE / "csv_unificado"

FIG_DIR = BASE / "figuras_incidencia"
TABLE_DIR = BASE / "tablas_incidencia"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

DENGUE_PATH = DATA_DIR / "dengue_2020_2026.csv"

df = pd.read_csv(
    DENGUE_PATH,
    parse_dates=["FECHA"]
)

df = df.set_index("FECHA").sort_index()
df

,SEXO,EDAD,REGION,GRUPO_EDAD,ESTADO,MUNICIPIO,LAT,LON
FECHA,,,,,,,,
2020-01-01,M,36,Region 1,30 a 39 anios,SINALOA,CULIACAN,24.662580,-107.259452
2020-01-01,M,25,Region 5,18 a 29 anios,OAXACA,SANTO DOMINGO DE MORELOS,15.833547,-96.634413
2020-01-01,H,22,Region 5,18 a 29 anios,OAXACA,SAN ANTONIO NANAHUATIPAM,18.146151,-97.164512
2020-01-01,M,14,Region 6,<18,YUCATAN,AKIL,20.262062,-89.348379
2020-01-01,H,77,Region 6,>=60 anios,YUCATAN,CELESTUN,20.888217,-90.278038
...,...,...,...,...,...,...,...,...
2026-05-18,H,8,Region 1,<18,BAJA CALIFORNIA SUR,LA PAZ,24.184341,-110.641288
2026-05-19,H,58,Region 4,50 a 59 anios,JALISCO,COCULA,20.390353,-103.824459
2026-05-19,M,21,Region 4,18 a 29 anios,JALISCO,COCULA,20.390353,-103.824459


## Series semanales de casos confirmados

Los registros individuales de dengue se agregan a frecuencia semanal para construir una serie temporal por entidad federativa.

Se utilizan semanas que terminan en domingo, mediante la frecuencia:

```python
"W-SUN"
```

De esta forma, cada fila representa una semana y cada columna un estado:

$$
\mathbf{Y}^{(\mathrm{casos})}=
\begin{pmatrix}
y_{1,1} & y_{2,1} & \cdots & y_{32,1}\\
y_{1,2} & y_{2,2} & \cdots & y_{32,2}\\
\vdots & \vdots & \ddots & \vdots \\
y_{1,T} & y_{2,T} & \cdots & y_{32,T}
\end{pmatrix}.
$$

Esta representación conserva los conteos originales y será útil posteriormente para modelos probabilísticos de conteo. También se verifica que la suma de todos los valores semanales coincida exactamente con el número total de registros confirmados del dataset original.



In [3]:
# Series temporales semanales de casos por estado

casos_semanales_estado = (
    df
    .groupby("ESTADO")
    .resample("W-SUN")
    .size()
    .unstack(level=0)
    .fillna(0)
    .astype(int)
)

casos_semanales_estado.index.name = "FECHA"
casos_semanales_estado.columns.name = None

print("Dimensiones:")
print(casos_semanales_estado.shape)

print("\nRango temporal:")
print(
    casos_semanales_estado.index.min(),
    "a",
    casos_semanales_estado.index.max()
)

print("\nNúmero de estados:")
print(casos_semanales_estado.shape[1])

Dimensiones:
(334, 32)

Rango temporal:
2020-01-05 00:00:00 a 2026-05-24 00:00:00

Número de estados:
32


In [4]:
casos_semanales_estado

,AGUASCALIENTES,BAJA CALIFORNIA,BAJA CALIFORNIA SUR,CAMPECHE,CHIAPAS,CHIHUAHUA,COAHUILA,COLIMA,DISTRITO FEDERAL,DURANGO,...,QUINTANA ROO,SAN LUIS POTOSI,SINALOA,SONORA,TABASCO,TAMAULIPAS,TLAXCALA,VERACRUZ,YUCATAN,ZACATECAS
FECHA,,,,,,,,,,,,,,,,,,,,,
2020-01-05,0,1,0,5,2,0,1,2,1,0,...,24,0,5,0,16,5,0,41,12,0
2020-01-12,0,1,1,5,12,1,1,4,2,0,...,33,0,12,2,21,6,0,69,10,0
2020-01-19,0,0,0,2,14,0,0,10,1,0,...,35,0,3,0,22,3,0,87,19,0
2020-01-26,0,0,1,7,8,0,0,2,0,0,...,28,0,11,1,30,10,0,66,21,0
2020-02-02,0,0,0,2,8,0,0,4,0,0,...,19,1,4,0,26,9,0,58,14,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-26,0,0,4,1,4,0,0,0,0,0,...,3,0,11,27,16,3,0,15,0,0
2026-05-03,0,0,6,0,9,0,0,0,0,0,...,4,0,14,38,14,0,0,13,0,0
2026-05-10,0,0,8,1,3,0,0,0,1,0,...,4,1,16,50,8,1,0,7,1,0


In [5]:
# Verificación de consistencia

print("Total de casos en df:")
print(len(df))

print("\nTotal en las series semanales:")
print(casos_semanales_estado.to_numpy().sum())

print("\nDiferencia:")
print(
    len(df)
    - casos_semanales_estado.to_numpy().sum()
)

Total de casos en df:
242999

Total en las series semanales:
242999

Diferencia:
0


In [6]:
# Guardar series temporales semanales

SERIES_DIR = BASE / "series_temporales"
SERIES_DIR.mkdir(parents=True, exist_ok=True)

SERIES_PATH = SERIES_DIR / "casos_semanales_dengue_por_estado_2020_2026.csv"

casos_semanales_estado.to_csv(
    SERIES_PATH,
    encoding="utf-8"
)

print("Archivo guardado en:")
print(SERIES_PATH.resolve())

Archivo guardado en:
/Users/juanalbertomartinez/Desktop/Dengue/datos_dengue_historicos/series_temporales/casos_semanales_dengue_por_estado_2020_2026.csv


## Incidencia semanal por 100,000 habitantes

Los conteos absolutos no son directamente comparables entre estados con tamaños poblacionales muy diferentes. Por esta razón, se construye una segunda representación basada en la incidencia semanal por 100,000 habitantes.

Para cada estado $s$, semana $t$ y año $a(t)$, se calcula:

$$
I_{s,t}=
\frac{C_{s,t}}
{N_{s,a(t)}}
\times 100,000,
$$

donde:

* $C_{s,t}$ es el número de casos confirmados durante la semana;
* $N_{s,a(t)}$ es la población estatal correspondiente al año de la observación.

La población cambia entre años, por lo que cada registro se cruza con la población correspondiente a su combinación estado--año antes de realizar la agregación semanal.

El resultado es una matriz:

$$
\mathbf{Y}^{(\mathrm{inc})}
\in
\mathbb{R}^{T\times 32},
$$

con el mismo índice temporal y el mismo orden espacial que la matriz de casos semanales.

La incidencia será la representación principal para comparar la **forma de las trayectorias epidémicas entre estados**, mientras que los casos absolutos se conservarán para el modelado probabilístico y para análisis de sensibilidad.



In [7]:
# Incidencia semanal de dengue por estado
# Casos por 100,000 habitantes


POB_PATH = Path("poblacion_estatal_2020_2026.csv")

pop = pd.read_csv(POB_PATH)

pop = pop.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

pop["ANIO"] = pd.to_numeric(
    pop["ANIO"],
    errors="coerce"
).astype(int)

pop["POBLACION"] = pd.to_numeric(
    pop["POBLACION"],
    errors="coerce"
)

print("Población:")
print(pop.shape)

print(pop.head())


df_inc = (
    df
    .reset_index()
    [["FECHA", "ESTADO"]]
    .copy()
)

df_inc["ANIO"] = df_inc["FECHA"].dt.year


df_inc = df_inc.merge(
    pop[
        [
            "ESTADO",
            "ANIO",
            "POBLACION"
        ]
    ],
    on=["ESTADO", "ANIO"],
    how="left"
)

print("\nValores faltantes después del cruce:")
print(df_inc.isna().sum())



df_inc["APORTE_INCIDENCIA_100K"] = (
    100_000 / df_inc["POBLACION"]
)



incidencia_semanal_estado = (
    df_inc
    .set_index("FECHA")
    .groupby("ESTADO")["APORTE_INCIDENCIA_100K"]
    .resample("W-SUN")
    .sum()
    .unstack(level=0)
    .fillna(0)
)

incidencia_semanal_estado.index.name = "FECHA"
incidencia_semanal_estado.columns.name = None


# Revisión
print("\nDimensiones:")
print(incidencia_semanal_estado.shape)

print("\nRango temporal:")
print(
    incidencia_semanal_estado.index.min(),
    "a",
    incidencia_semanal_estado.index.max()
)

print("\nNúmero de estados:")
print(incidencia_semanal_estado.shape[1])

print("\nValores faltantes:")
print(incidencia_semanal_estado.isna().sum().sum())

Población:
(224, 3)
                ESTADO  ANIO  POBLACION
0       AGUASCALIENTES  2020    1456050
1      BAJA CALIFORNIA  2020    3838619
2  BAJA CALIFORNIA SUR  2020     815098
3             CAMPECHE  2020     939179
4              CHIAPAS  2020    5715260

Valores faltantes después del cruce:
FECHA        0
ESTADO       0
ANIO         0
POBLACION    0
dtype: int64

Dimensiones:
(334, 32)

Rango temporal:
2020-01-05 00:00:00 a 2026-05-24 00:00:00

Número de estados:
32

Valores faltantes:
0


In [8]:
incidencia_semanal_estado

,AGUASCALIENTES,BAJA CALIFORNIA,BAJA CALIFORNIA SUR,CAMPECHE,CHIAPAS,CHIHUAHUA,COAHUILA,COLIMA,DISTRITO FEDERAL,DURANGO,...,QUINTANA ROO,SAN LUIS POTOSI,SINALOA,SONORA,TABASCO,TAMAULIPAS,TLAXCALA,VERACRUZ,YUCATAN,ZACATECAS
FECHA,,,,,,,,,,,,,,,,,,,,,
2020-01-05,0.0,0.026051,0.000000,0.532380,0.034994,0.000000,0.031278,0.268412,0.010715,0.0,...,1.264328,0.000000,0.162350,0.000000,0.655617,0.139045,0.0,0.501880,0.507803,0.0
2020-01-12,0.0,0.026051,0.122685,0.532380,0.209964,0.026229,0.031278,0.536825,0.021430,0.0,...,1.738452,0.000000,0.389639,0.066616,0.860497,0.166853,0.0,0.844627,0.423169,0.0
2020-01-19,0.0,0.000000,0.000000,0.212952,0.244958,0.000000,0.000000,1.342062,0.010715,0.0,...,1.843812,0.000000,0.097410,0.000000,0.901473,0.083427,0.0,1.064964,0.804021,0.0
2020-01-26,0.0,0.000000,0.122685,0.745332,0.139976,0.000000,0.000000,0.268412,0.000000,0.0,...,1.475050,0.000000,0.357169,0.033308,1.229282,0.278089,0.0,0.807904,0.888655,0.0
2020-02-02,0.0,0.000000,0.000000,0.212952,0.139976,0.000000,0.000000,0.536825,0.000000,0.0,...,1.000927,0.034823,0.129880,0.000000,1.065378,0.250280,0.0,0.709976,0.592436,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-26,0.0,0.000000,0.433965,0.103949,0.064668,0.000000,0.000000,0.000000,0.000000,0.0,...,0.140116,0.000000,0.342889,0.852105,0.647074,0.079418,0.0,0.184916,0.000000,0.0
2026-05-03,0.0,0.000000,0.650947,0.000000,0.145502,0.000000,0.000000,0.000000,0.000000,0.0,...,0.186821,0.000000,0.436404,1.199259,0.566190,0.000000,0.0,0.160260,0.000000,0.0
2026-05-10,0.0,0.000000,0.867929,0.103949,0.048501,0.000000,0.000000,0.000000,0.010917,0.0,...,0.186821,0.033304,0.498747,1.577972,0.323537,0.026473,0.0,0.086294,0.039331,0.0


In [9]:
INC_SEMANAL_PATH = (
    SERIES_DIR
    / "incidencia_semanal_dengue_por_estado_2020_2026.csv"
)

incidencia_semanal_estado.to_csv(
    INC_SEMANAL_PATH,
    encoding="utf-8"
)


# Covariables meteorológicas exógenas

La dinámica temporal del dengue puede estar relacionada con condiciones ambientales que afectan el desarrollo del vector, la disponibilidad de criaderos y la velocidad de los procesos biológicos asociados con la transmisión.

Para incorporar esta información se consideran covariables meteorológicas de NASA POWER. En este proyecto se utilizan inicialmente nueve variables:

| Prioridad          | ID NASA POWER       | Variable                      | Papel en el modelo                                                      |
| ------------------ | ------------------- | ----------------------------- | ----------------------------------------------------------------------- |
| **Esencial**       | `T2M`               | Temperatura media a 2 m       | Nivel térmico general                                                   |
| **Esencial**       | `T2M_MAX`           | Temperatura máxima            | Extremos cálidos y no linealidad                                        |
| **Esencial**       | `T2M_MIN`           | Temperatura mínima            | Condiciones térmicas nocturnas                                          |
| **Esencial**       | `RH2M`              | Humedad relativa a 2 m        | Humedad ambiental                                                       |
| **Esencial**       | `PRECTOTCORR`       | Precipitación corregida       | Formación y persistencia de hábitats acuáticos                          |
| **Esencial**       | `ALLSKY_SFC_SW_DWN` | Radiación solar superficial   | Energía, calentamiento y secado ambiental                               |
| **Importante**     | `WS2M`              | Velocidad del viento a 2 m    | Dinámica atmosférica y posible modificación de actividad/dispersión     |
| **Importante**     | `GWETTOP`           | Humedad superficial del suelo | Memoria hidrológica de corto plazo                                      |
| **Complementaria** | `EVLAND`            | Evaporación terrestre         | Pérdida de humedad y persistencia de condiciones húmedas                |



La literatura de pronóstico de dengue ha encontrado utilidad predictiva en temperatura y precipitación, y también muestra que las asociaciones climáticas pueden ser retardadas y no lineales. Por ejemplo, existen estudios con efectos térmicos observados en rezagos de 0–4 semanas y efectos de precipitación intensa alrededor de 3–7 semanas en determinados contextos; no hay que asumir que esos rezagos son universales para México, pero sí justifican que los probemos sistemáticamente.

https://journals.plos.org/plosntds/article?id=10.1371/journal.pntd.0001908

https://www.cambridge.org/core/journals/epidemiology-and-infection/article/lag-effect-of-climatic-variables-on-dengue-burden-in-india/D9AC458913A43934395A64AC23BA3233

https://www.frontiersin.org/journals/public-health/articles/10.3389/fpubh.2023.1287678/full

Estas variables no se utilizarán necesariamente de manera simultánea en el modelo final. Antes del pronóstico se estudiarán:

- redundancia entre covariables;
- correlaciones contemporáneas y retardadas;
- posibles transformaciones;
- efectos de rezagos;
- estabilidad de la contribución predictiva.

El objetivo no es utilizar el mayor número posible de predictores, sino construir un conjunto parsimonioso de covariables exógenas con capacidad explicativa y predictiva.

In [19]:
# NASA POWER: descarga diaria de covariables meteorológicas

FECHA_INICIO = df.index.min().strftime("%Y%m%d")

FECHA_FIN_POWER = (
    df.index.max()
    + pd.offsets.Week(weekday=6)
)

FECHA_FIN = FECHA_FIN_POWER.strftime("%Y%m%d")

print("Última fecha de casos:")
print(df.index.max())

print("\nÚltimo día solicitado a NASA POWER:")
print(FECHA_FIN_POWER)

print("\nPeriodo de descarga:")
print(FECHA_INICIO, "a", FECHA_FIN)

print("Periodo de descarga NASA POWER:")
print(FECHA_INICIO, "a", FECHA_FIN)



# Variables seleccionadas
PARAMETROS_POWER = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "PRECTOTCORR",
    "ALLSKY_SFC_SW_DWN",
    "WS2M",
    "GWETTOP",
    "EVLAND",
]



print("\nNúmero de variables:")
print(len(PARAMETROS_POWER))

print("\nVariables:")
for var in PARAMETROS_POWER:
    print("-", var)



POWER_DIR = BASE / "nasa_power"

RAW_DIR = POWER_DIR / "diario_por_estado"
MATRIX_DIR = POWER_DIR / "matrices_diarias"

RAW_DIR.mkdir(parents=True, exist_ok=True)
MATRIX_DIR.mkdir(parents=True, exist_ok=True)


POWER_URL = (
    "https://power.larc.nasa.gov"
    "/api/temporal/daily/point"
)

Última fecha de casos:
2026-05-20 00:00:00

Último día solicitado a NASA POWER:
2026-05-24 00:00:00

Periodo de descarga:
20200101 a 20260524
Periodo de descarga NASA POWER:
20200101 a 20260524

Número de variables:
9

Variables:
- T2M
- T2M_MAX
- T2M_MIN
- RH2M
- PRECTOTCORR
- ALLSKY_SFC_SW_DWN
- WS2M
- GWETTOP
- EVLAND


## Representación espacial de los estados

Para descargar series meteorológicas se necesita asociar cada entidad federativa con una localización geográfica.

En esta primera etapa del proyecto se utiliza un **punto representativo interior** para cada estado. A partir de las geometrías estatales se obtiene un punto garantizado dentro del polígono mediante:

```python
representative_point()
```

Posteriormente, sus coordenadas de latitud y longitud se utilizan para consultar NASA POWER.

Este procedimiento proporciona una representación meteorológica simplificada de cada entidad y permite construir de manera eficiente el pipeline completo de descarga, agregación temporal, clustering y pronóstico.

Debe tenerse presente que esta aproximación representa cada estado mediante un solo punto. En análisis posteriores se podrá evaluar una estrategia espacial más detallada, especialmente para entidades extensas o con fuerte heterogeneidad climática interna.

Antes de realizar las descargas también se homologan los nombres estatales entre:

* el GeoJSON;
* la base de dengue;
* la base de población.

Esto garantiza que las matrices epidemiológicas y climáticas puedan combinarse sin pérdidas por diferencias de nomenclatura.



In [20]:
# Cargar geometrías y obtener coordenadas


URL_GEOJSON = (
    "https://raw.githubusercontent.com/"
    "strotgen/mexico-leaflet/refs/heads/master/states.geojson"
)

gdf = gpd.read_file(URL_GEOJSON)

print("Columnas originales del GeoDataFrame:")
print(gdf.columns.tolist())


# homologar nombres

def quitar_acentos_mayusculas(texto):
    """
    Convierte texto a mayúsculas, elimina acentos y normaliza espacios.
    """
    
    if pd.isna(texto):
        return texto

    texto = str(texto).strip().upper()

    texto = unicodedata.normalize(
        "NFD",
        texto
    )

    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto


def homologar_estado_geojson(texto):
    """
    Homologa los nombres del GeoJSON con los usados
    en el dataset de dengue.
    """
    
    texto = quitar_acentos_mayusculas(texto)

    reemplazos = {
        "COAHUILA DE ZARAGOZA": "COAHUILA",
        "MICHOACAN DE OCAMPO": "MICHOACAN",
        "VERACRUZ DE IGNACIO DE LA LLAVE": "VERACRUZ",
        "DISTRITO FEDERAL": "DISTRITO FEDERAL",
        "MEXICO": "MEXICO",
        "QUERETARO": "QUERETARO",
        "NUEVO LEON": "NUEVO LEON",
        "SAN LUIS POTOSI": "SAN LUIS POTOSI",
        "YUCATAN": "YUCATAN",
    }

    return reemplazos.get(
        texto,
        texto
    )


gdf["ESTADO"] = (
    gdf["state_name"]
    .apply(homologar_estado_geojson)
)

print("\nNombres homologados:")
display(
    gdf[
        [
            "state_name",
            "ESTADO"
        ]
    ]
    .sort_values("ESTADO")
    .reset_index(drop=True)
)



# Obtener punto representativo interior


gdf_power = gdf.copy()

gdf_power["PUNTO_POWER"] = (
    gdf_power.geometry.representative_point()
)

gdf_power["LAT_POWER"] = (
    gdf_power["PUNTO_POWER"].y
)

gdf_power["LON_POWER"] = (
    gdf_power["PUNTO_POWER"].x
)



# Tabla final de coordenadas
coordenadas_power = (
    gdf_power[
        [
            "ESTADO",
            "LAT_POWER",
            "LON_POWER"
        ]
    ]
    .sort_values("ESTADO")
    .reset_index(drop=True)
)

print("\nPuntos de descarga NASA POWER:")
display(coordenadas_power)

print("\nNúmero de estados:")
print(
    coordenadas_power["ESTADO"].nunique()
)


# Guardar coordenadas para reproducibilidad

coordenadas_power.to_csv(
    POWER_DIR / "coordenadas_representativas_estados.csv",
    index=False,
    encoding="utf-8"
)

print("\nCoordenadas guardadas en:")
print(
    (
        POWER_DIR
        / "coordenadas_representativas_estados.csv"
    ).resolve()
)

Columnas originales del GeoDataFrame:
['id', 'state_code', 'state_name', 'geometry']

Nombres homologados:


,state_name,ESTADO
0,Aguascalientes,AGUASCALIENTES
1,Baja California,BAJA CALIFORNIA
2,Baja California Sur,BAJA CALIFORNIA SUR
3,Campeche,CAMPECHE
4,Chiapas,CHIAPAS
5,Chihuahua,CHIHUAHUA
6,Coahuila de Zaragoza,COAHUILA
7,Colima,COLIMA
8,Distrito Federal,DISTRITO FEDERAL
9,Durango,DURANGO



Puntos de descarga NASA POWER:


,ESTADO,LAT_POWER,LON_POWER
0,AGUASCALIENTES,22.035219,-102.320532
1,BAJA CALIFORNIA,30.352741,-115.247277
2,BAJA CALIFORNIA SUR,25.433774,-111.560536
3,CAMPECHE,19.330942,-89.944849
4,CHIAPAS,16.256292,-92.238030
5,CHIHUAHUA,28.690655,-106.041030
6,COAHUILA,27.204344,-102.311584
7,COLIMA,19.098242,-103.926938
8,DISTRITO FEDERAL,19.314301,-99.156661
9,DURANGO,24.552445,-104.572277



Número de estados:
32

Coordenadas guardadas en:
/Users/juanalbertomartinez/Desktop/Dengue/datos_dengue_historicos/nasa_power/coordenadas_representativas_estados.csv


## Descarga de series meteorológicas diarias

Las covariables meteorológicas se descargan inicialmente a frecuencia diaria para conservar la mayor resolución temporal disponible antes de construir las series semanales.

Para cada estado se realiza una consulta utilizando:

* el punto representativo de la entidad;
* el periodo completo de estudio;
* las variables meteorológicas seleccionadas.

La descarga se realiza con un mecanismo de reintentos automáticos ante errores temporales del servidor. Además, cada estado se guarda individualmente inmediatamente después de su descarga.

Esta estrategia tiene dos ventajas:

1. permite reanudar el proceso sin repetir solicitudes ya completadas;
2. facilita detectar y corregir errores de descarga para estados específicos.

La fecha final solicitada se extiende hasta el domingo de cierre de la última semana epidemiológica analizada. Esto permite que las covariables meteorológicas y las series de dengue tengan exactamente el mismo soporte temporal al realizar la agregación semanal.



In [21]:
# Función para descargar NASA POWER


retry_strategy = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()
session.mount("https://", adapter)



# Función de descarga
def descargar_power_estado(
    estado,
    latitud,
    longitud,
    fecha_inicio,
    fecha_fin,
    parametros,
    session
):
    """
    Descarga datos diarios NASA POWER para un punto geográfico.
    """

    params = {
        "parameters": ",".join(parametros),
        "community": "AG",
        "longitude": longitud,
        "latitude": latitud,
        "start": fecha_inicio,
        "end": fecha_fin,
        "format": "JSON",
        "time-standard": "LST",
    }

    response = session.get(
        POWER_URL,
        params=params,
        timeout=180
    )

    response.raise_for_status()

    payload = response.json()


    # Extraer parámetros
    try:
        datos = payload["properties"]["parameter"]
    except KeyError as exc:
        raise RuntimeError(
            f"Respuesta inesperada para {estado}: "
            f"{payload.keys()}"
        ) from exc


    # Construir DataFrame
    datos_estado = pd.DataFrame(datos)

    datos_estado.index = pd.to_datetime(
        datos_estado.index,
        format="%Y%m%d"
    )

    datos_estado.index.name = "FECHA"

    datos_estado = datos_estado.sort_index()


    # Valor de relleno

    fill_value = (
        payload
        .get("header", {})
        .get("fill_value", -999)
    )

    datos_estado = datos_estado.replace(
        fill_value,
        np.nan
    )

    # Por seguridad
    datos_estado = datos_estado.replace(
        -999,
        np.nan
    )

    # Verificar variables
    faltantes = sorted(
        set(parametros)
        - set(datos_estado.columns)
    )

    if faltantes:
        raise ValueError(
            f"{estado}: variables ausentes en respuesta: "
            f"{faltantes}"
        )

    datos_estado = datos_estado[parametros]

    # Metadatos espaciales
    datos_estado.insert(
        0,
        "ESTADO",
        estado
    )

    datos_estado.insert(
        1,
        "LAT_POWER",
        latitud
    )

    datos_estado.insert(
        2,
        "LON_POWER",
        longitud
    )

    return datos_estado.reset_index()

In [22]:
# Descargar las series diarias para los 32 estados

resultados = []
errores = []

n_estados = len(coordenadas_power)

for i, row in coordenadas_power.iterrows():

    estado = row["ESTADO"]
    latitud = row["LAT_POWER"]
    longitud = row["LON_POWER"]

    nombre_estado = (
        estado
        .lower()
        .replace(" ", "_")
    )

    archivo_estado = (
        RAW_DIR
        / f"power_diario_{nombre_estado}.csv"
    )

    print(
        f"[{i + 1:02d}/{n_estados}] {estado}"
    )

    try:

        if archivo_estado.exists():

            print("    archivo existente: se reutiliza")

            datos_estado = pd.read_csv(
                archivo_estado,
                parse_dates=["FECHA"]
            )

        else:

            print("    descargando...")

            datos_estado = descargar_power_estado(
                estado=estado,
                latitud=latitud,
                longitud=longitud,
                fecha_inicio=FECHA_INICIO,
                fecha_fin=FECHA_FIN,
                parametros=PARAMETROS_POWER,
                session=session
            )

            datos_estado.to_csv(
                archivo_estado,
                index=False,
                encoding="utf-8"
            )

            # Pausa entre solicitudes
            time.sleep(1)

        resultados.append(datos_estado)

        print(
            f"    filas: {len(datos_estado):,}"
        )

    except Exception as exc:

        print(f"    ERROR: {exc}")

        errores.append(
            {
                "ESTADO": estado,
                "ERROR": str(exc)
            }
        )


print("\nProceso terminado.")

print("\nEstados descargados:")
print(len(resultados))

print("\nErrores:")
print(len(errores))

if errores:
    display(pd.DataFrame(errores))

[01/32] AGUASCALIENTES
    archivo existente: se reutiliza
    filas: 2,336
[02/32] BAJA CALIFORNIA
    archivo existente: se reutiliza
    filas: 2,336
[03/32] BAJA CALIFORNIA SUR
    archivo existente: se reutiliza
    filas: 2,336
[04/32] CAMPECHE
    archivo existente: se reutiliza
    filas: 2,336
[05/32] CHIAPAS
    archivo existente: se reutiliza
    filas: 2,336
[06/32] CHIHUAHUA
    archivo existente: se reutiliza
    filas: 2,336
[07/32] COAHUILA
    archivo existente: se reutiliza
    filas: 2,336
[08/32] COLIMA
    archivo existente: se reutiliza
    filas: 2,336
[09/32] DISTRITO FEDERAL
    archivo existente: se reutiliza
    filas: 2,336
[10/32] DURANGO
    archivo existente: se reutiliza
    filas: 2,336
[11/32] GUANAJUATO
    archivo existente: se reutiliza
    filas: 2,336
[12/32] GUERRERO
    archivo existente: se reutiliza
    filas: 2,336
[13/32] HIDALGO
    archivo existente: se reutiliza
    filas: 2,336
[14/32] JALISCO
    archivo existente: se reutiliza
    fila

## Integración y control de calidad de las series diarias

Una vez descargadas las series correspondientes a los 32 estados, los archivos individuales se integran en un único dataset diario en formato largo.

La estructura general es:

$$
(\text{FECHA},\text{ESTADO},
X^{(1)},X^{(2)},\ldots,X^{(p)}).
$$

Antes de continuar con la agregación semanal se revisan:

* número de estados disponibles;
* rango temporal;
* número total de observaciones;
* presencia de valores faltantes;
* disponibilidad de todas las variables solicitadas.

Mantener una copia del dataset diario es importante porque permite reproducir posteriormente otras reglas de agregación temporal sin necesidad de repetir las consultas a la fuente original.



In [23]:
# Integrar los estados en un único dataset diario

if len(resultados) != 32:
    print(
        "ADVERTENCIA:",
        f"se tienen {len(resultados)} de 32 estados."
    )


power_diario = pd.concat(
    resultados,
    ignore_index=True
)

power_diario = power_diario.sort_values(
    ["FECHA", "ESTADO"]
).reset_index(drop=True)


# Revisión
print("Dimensiones:")
print(power_diario.shape)

print("\nRango temporal:")
print(
    power_diario["FECHA"].min(),
    "a",
    power_diario["FECHA"].max()
)

print("\nEstados:")
print(power_diario["ESTADO"].nunique())

print("\nValores faltantes por variable:")
print(
    power_diario[PARAMETROS_POWER]
    .isna()
    .sum()
)

Dimensiones:
(74752, 13)

Rango temporal:
2020-01-01 00:00:00 a 2026-05-24 00:00:00

Estados:
32

Valores faltantes por variable:
T2M                  0
T2M_MAX              0
T2M_MIN              0
RH2M                 0
PRECTOTCORR          0
ALLSKY_SFC_SW_DWN    0
WS2M                 0
GWETTOP              0
EVLAND               0
dtype: int64


In [24]:
power_diario

,FECHA,ESTADO,LAT_POWER,LON_POWER,T2M,T2M_MAX,T2M_MIN,RH2M,PRECTOTCORR,ALLSKY_SFC_SW_DWN,WS2M,GWETTOP,EVLAND
0,2020-01-01,AGUASCALIENTES,22.035219,-102.320532,13.29,20.08,7.35,68.48,6.93,10.19,3.90,0.46,1.54
1,2020-01-01,BAJA CALIFORNIA,30.352741,-115.247277,9.72,18.60,3.09,50.68,0.00,14.03,2.26,0.43,0.63
2,2020-01-01,BAJA CALIFORNIA SUR,25.433774,-111.560536,14.96,19.79,11.70,81.29,0.28,14.42,1.79,0.57,1.23
3,2020-01-01,CAMPECHE,19.330942,-89.944849,24.68,30.06,20.69,79.32,0.52,15.61,0.02,0.58,3.31
4,2020-01-01,CHIAPAS,16.256292,-92.238030,18.27,25.76,12.75,81.27,0.16,16.94,0.96,0.64,2.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...
74747,2026-05-24,TAMAULIPAS,24.960919,-98.358241,29.29,36.02,25.20,70.78,9.07,13.95,1.96,0.57,4.38
74748,2026-05-24,TLAXCALA,19.409963,-98.107904,17.69,23.99,13.69,72.61,3.18,17.69,0.72,0.49,2.57
74749,2026-05-24,VERACRUZ,19.781609,-96.870858,27.28,32.38,23.33,70.28,1.62,23.46,1.99,0.64,3.26
74750,2026-05-24,YUCATAN,20.604650,-89.090997,31.36,38.46,26.69,63.87,4.60,22.43,0.00,0.38,1.45


In [25]:
# Guardar dataset diario completo

POWER_DIARIO_PATH = (
    POWER_DIR
    / "nasa_power_diario_estados_2020_2026.csv"
)

power_diario.to_csv(
    POWER_DIARIO_PATH,
    index=False,
    encoding="utf-8"
)

print("Dataset guardado en:")
print(POWER_DIARIO_PATH.resolve())

Dataset guardado en:
/Users/juanalbertomartinez/Desktop/Dengue/datos_dengue_historicos/nasa_power/nasa_power_diario_estados_2020_2026.csv


## Agregación de las covariables meteorológicas a frecuencia semanal

Las variables meteorológicas diarias se agregan utilizando las mismas semanas que las series de dengue:

```python
"W-SUN"
```

La regla de agregación depende de la naturaleza de cada variable.

Para temperatura media, humedad, radiación, viento, humedad del suelo y evaporación se utiliza inicialmente el promedio semanal:

$$
\overline{X}_{s,t}=
\frac{1}{n_t}
\sum_{d\in t}X_{s,d}.
$$

Para la precipitación se utiliza la suma semanal:

$$
P_{s,t}=
\sum_{d\in t}P_{s,d}.
$$

Para las temperaturas máxima y mínima se conservan los extremos semanales:

$$
T^{\max}_{s,t}
=\max_{d\in t}T^{\max}_{s,d},
$$

$$
T^{\min}_{s,t}
=\min_{d\in t}T^{\min}_{s,d}.
$$

Estas reglas permiten obtener para cada covariable una serie temporal con la misma frecuencia y el mismo índice que los casos y la incidencia de dengue.

En etapas posteriores podrá evaluarse la utilidad de representaciones adicionales, por ejemplo:

* media semanal de las máximas diarias;
* media semanal de las mínimas diarias;
* número de días con lluvia;
* precipitación máxima diaria semanal;
* acumulados móviles;
* anomalías climáticas;
* rezagos distribuidos.


In [34]:
# Agregar NASA POWER a frecuencia semanal y crear una matriz semanal por variable

AGREGACION_SEMANAL = {
    # Temperatura
    "T2M": "mean",
    "T2M_MAX": "max",
    "T2M_MIN": "min",

    # Humedad
    "RH2M": "mean",

    # Precipitación
    "PRECTOTCORR": "sum",

    # Radiación
    "ALLSKY_SFC_SW_DWN": "mean",

    # Viento
    "WS2M": "mean",

    # Humedad superficial
    "GWETTOP": "mean",

    # Evaporación
    # Por ahora usamos media semanal.
    # Podemos revisar la unidad antes del modelado final.
    "EVLAND": "mean",
}

print("Reglas de agregación semanal:")
for variable, regla in AGREGACION_SEMANAL.items():
    print(f"{variable:25s} -> {regla}")

Reglas de agregación semanal:
T2M                       -> mean
T2M_MAX                   -> max
T2M_MIN                   -> min
RH2M                      -> mean
PRECTOTCORR               -> sum
ALLSKY_SFC_SW_DWN         -> mean
WS2M                      -> mean
GWETTOP                   -> mean
EVLAND                    -> mean


In [35]:
power_diario.columns

Index(['FECHA', 'ESTADO', 'LAT_POWER', 'LON_POWER', 'T2M', 'T2M_MAX',
       'T2M_MIN', 'RH2M', 'PRECTOTCORR', 'ALLSKY_SFC_SW_DWN', 'WS2M',
       'GWETTOP', 'EVLAND'],
      dtype='object')

In [37]:
# Agregar el dataset diario a semanas W-SUN


power_semanal = (
    power_diario
    .set_index("FECHA")
    .groupby("ESTADO")
    .resample("W-SUN")
    .agg(AGREGACION_SEMANAL)
    .reset_index()
)

power_semanal = (
    power_semanal
    .sort_values(["FECHA", "ESTADO"])
    .reset_index(drop=True)
)

print("Dimensiones del dataset semanal:")
print(power_semanal.shape)

print("\nRango temporal:")
print(
    power_semanal["FECHA"].min(),
    "a",
    power_semanal["FECHA"].max()
)

print("\nNúmero de estados:")
print(power_semanal["ESTADO"].nunique())

print("\nNúmero de semanas:")
print(power_semanal["FECHA"].nunique())

print("\nValores faltantes por variable:")
print(
    power_semanal[PARAMETROS_POWER]
    .isna()
    .sum()
)

Dimensiones del dataset semanal:
(10688, 11)

Rango temporal:
2020-01-05 00:00:00 a 2026-05-24 00:00:00

Número de estados:
32

Número de semanas:
334

Valores faltantes por variable:
T2M                  0
T2M_MAX              0
T2M_MIN              0
RH2M                 0
PRECTOTCORR          0
ALLSKY_SFC_SW_DWN    0
WS2M                 0
GWETTOP              0
EVLAND               0
dtype: int64


In [39]:
power_semanal

,ESTADO,FECHA,T2M,T2M_MAX,T2M_MIN,RH2M,PRECTOTCORR,ALLSKY_SFC_SW_DWN,WS2M,GWETTOP,EVLAND
0,AGUASCALIENTES,2020-01-05,8.984000,20.08,-2.65,56.996000,13.63,16.994000,3.206000,0.444000,1.230000
1,BAJA CALIFORNIA,2020-01-05,10.624000,22.74,3.09,45.482000,0.00,14.320000,2.440000,0.424000,0.656000
2,BAJA CALIFORNIA SUR,2020-01-05,15.144000,23.73,9.90,77.304000,0.28,15.738000,2.718000,0.558000,1.464000
3,CAMPECHE,2020-01-05,24.450000,32.08,18.35,74.244000,3.14,14.510000,0.038000,0.556000,2.810000
4,CHIAPAS,2020-01-05,16.622000,25.76,10.07,83.394000,2.59,14.266000,1.960000,0.638000,2.538000
...,...,...,...,...,...,...,...,...,...,...,...
10683,TAMAULIPAS,2026-05-24,28.880000,37.36,23.08,73.937143,64.19,17.824286,3.121429,0.537143,3.335714
10684,TLAXCALA,2026-05-24,17.804286,27.13,11.13,69.071429,27.89,19.058571,1.091429,0.462857,2.331429
10685,VERACRUZ,2026-05-24,27.155714,34.43,22.24,71.731429,21.98,22.927143,2.177143,0.637143,3.032857
10686,YUCATAN,2026-05-24,32.052857,39.94,25.18,55.357143,6.01,23.867143,0.000000,0.422857,1.060000


In [40]:
# Guardar dataset semanal en formato largo
POWER_SEMANAL_PATH = (
    POWER_DIR
    / "nasa_power_semanal_estados_2020_2026.csv"
)

power_semanal.to_csv(
    POWER_SEMANAL_PATH,
    index=False,
    encoding="utf-8"
)

print("Dataset semanal guardado en:")
print(POWER_SEMANAL_PATH.resolve())

Dataset semanal guardado en:
/Users/juanalbertomartinez/Desktop/Dengue/datos_dengue_historicos/nasa_power/nasa_power_semanal_estados_2020_2026.csv


## Construcción y alineación de matrices semanales

A partir del dataset climático semanal en formato largo se construye una matriz por covariable.

Cada matriz tiene la forma:

$$
\mathbf{X}^{(j)}
=\begin{pmatrix}
x^{(j)}_{1,1} & \cdots & x^{(j)}_{32,1}\\
x^{(j)}_{1,2} & \cdots & x^{(j)}_{32,2}\\
\vdots & \ddots & \vdots \\
x^{(j)}_{1,T} & \cdots & x^{(j)}_{32,T}
\end{pmatrix},
$$

donde:

* las filas corresponden a semanas;
* las columnas corresponden a estados;
* $j$ identifica la covariable meteorológica.

Antes de utilizar estas matrices se verifica explícitamente que coincidan con las series epidemiológicas en:

1. número de semanas;
2. fechas inicial y final;
3. nombres de los estados;
4. número de entidades;
5. ausencia de semanas o estados desalineados.

La alineación exacta es indispensable porque los análisis posteriores combinarán observaciones de la forma:

$$
\left(
Y_{s,t},
X^{(1)}_{s,t},
\ldots,
X^{(p)}_{s,t}
\right).
$$

El objetivo final es disponer de una estructura común para estudiar:

$$
\text{clustering dinámico de trayectorias}
$$

y posteriormente

$$
\text{pronóstico probabilístico por clúster}
+
\text{covariables exógenas rezagadas}.
$$

In [41]:
# Crear matrices semanales:
# filas = semanas
# columnas = estados

MATRIX_WEEKLY_DIR = POWER_DIR / "matrices_semanales"
MATRIX_WEEKLY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

series_semanales_power = {}

for variable in PARAMETROS_POWER:

    matriz = (
        power_semanal
        .pivot(
            index="FECHA",
            columns="ESTADO",
            values=variable
        )
        .sort_index()
    )

    matriz.index.name = "FECHA"
    matriz.columns.name = None

    series_semanales_power[variable] = matriz

    output_path = (
        MATRIX_WEEKLY_DIR
        / f"{variable.lower()}_semanal_por_estado_2020_2026.csv"
    )

    matriz.to_csv(
        output_path,
        encoding="utf-8"
    )

    print(
        f"{variable:25s}",
        "dimensiones:",
        matriz.shape,
        "| faltantes:",
        matriz.isna().sum().sum()
    )

T2M                       dimensiones: (334, 32) | faltantes: 0
T2M_MAX                   dimensiones: (334, 32) | faltantes: 0
T2M_MIN                   dimensiones: (334, 32) | faltantes: 0
RH2M                      dimensiones: (334, 32) | faltantes: 0
PRECTOTCORR               dimensiones: (334, 32) | faltantes: 0
ALLSKY_SFC_SW_DWN         dimensiones: (334, 32) | faltantes: 0
WS2M                      dimensiones: (334, 32) | faltantes: 0
GWETTOP                   dimensiones: (334, 32) | faltantes: 0
EVLAND                    dimensiones: (334, 32) | faltantes: 0


In [42]:
# Verificar alineación temporal con la incidencia semanal


print("Incidencia semanal:")
print(incidencia_semanal_estado.shape)

print("\nNASA POWER semanal, ejemplo T2M:")
print(series_semanales_power["T2M"].shape)

print("\nRango incidencia:")
print(
    incidencia_semanal_estado.index.min(),
    "a",
    incidencia_semanal_estado.index.max()
)

print("\nRango T2M:")
print(
    series_semanales_power["T2M"].index.min(),
    "a",
    series_semanales_power["T2M"].index.max()
)

# Comparar fechas
fechas_inc = set(
    incidencia_semanal_estado.index
)

fechas_power = set(
    series_semanales_power["T2M"].index
)

print("\nSemanas en incidencia que no están en POWER:")
print(sorted(fechas_inc - fechas_power))

print("\nSemanas en POWER que no están en incidencia:")
print(sorted(fechas_power - fechas_inc))


# Comparar estados

estados_inc = set(
    incidencia_semanal_estado.columns
)

estados_power = set(
    series_semanales_power["T2M"].columns
)

print("\nEstados en incidencia que no están en POWER:")
print(sorted(estados_inc - estados_power))

print("\nEstados en POWER que no están en incidencia:")
print(sorted(estados_power - estados_inc))

Incidencia semanal:
(334, 32)

NASA POWER semanal, ejemplo T2M:
(334, 32)

Rango incidencia:
2020-01-05 00:00:00 a 2026-05-24 00:00:00

Rango T2M:
2020-01-05 00:00:00 a 2026-05-24 00:00:00

Semanas en incidencia que no están en POWER:
[]

Semanas en POWER que no están en incidencia:
[]

Estados en incidencia que no están en POWER:
[]

Estados en POWER que no están en incidencia:
[]


## Resumen

En este notebook se construyeron las series temporales semanales que servirán como base para las siguientes etapas del proyecto.

A partir de los registros individuales de dengue se generaron dos matrices epidemiológicas:

* casos confirmados semanales por estado;
* incidencia semanal por 100,000 habitantes.

Ambas matrices contienen las mismas semanas y las mismas 32 entidades federativas.

Posteriormente se descargaron y procesaron covariables meteorológicas diarias de NASA POWER. Las series se agregaron a frecuencia semanal utilizando reglas acordes con la naturaleza de cada variable y se transformaron en matrices estado--semana.

La etapa final verificó que las series epidemiológicas y meteorológicas presentan una correspondencia exacta en:

* rango temporal;
* número de semanas;
* número de estados;
* nombres de las entidades.

De esta manera se obtuvo una colección de series multivariadas sincronizadas:

$$
Y_{s,t},
\qquad
X^{(1)}_{s,t},
\ldots,
X^{(p)}_{s,t},
$$

que permitirán desarrollar las dos etapas principales del proyecto:

$$
\boxed{
\text{Clustering dinámico de las trayectorias estatales}
}
$$

y

$$
\boxed{
\text{Pronóstico probabilístico por clúster con covariables exógenas}
}.
$$
